## Token Manager
Caches and exchanges agentic tokens for authenticated agent calls.

In [1]:
from __future__ import annotations

import asyncio
import base64
import binascii
import json
import logging
import time
from collections.abc import Sequence
from dataclasses import dataclass
from os import environ
from typing import Any

logger = logging.getLogger(__name__)

TOKEN_REFRESH_BUFFER_SECONDS = int(environ.get("TOKEN_REFRESH_BUFFER_SECONDS", "300"))

Stores a cached token and the metadata needed to validate it.

In [2]:
@dataclass(frozen=True)
class CachedToken:

    token: str
    expires_at: float
    scopes: tuple[str, ...]
    tenant_id: str | None

    def is_valid_for(self, scopes: tuple[str, ...]) -> bool:
        """Return whether the cached token is valid for the requested scopes."""
        if self.scopes and scopes and self.scopes != scopes:
            return False
        return self.expires_at - time.time() > TOKEN_REFRESH_BUFFER_SECONDS

In [3]:
_agentic_token_cache: dict[str, CachedToken] = {}
_agent_locks: dict[str, asyncio.Lock] = {}

Return a cached token or start an authenticated token exchange.

In [4]:
def get_token(
    *,
    agent_id: str | None,
    tenant_id: str | None = None,
    auth: Any | None = None,
    auth_handler_name: str | None = None,
    context: Any | None = None,
    scopes: Sequence[str] | str | None = None,
):
    normalized_scopes = _normalize_scopes(scopes)
    if auth is not None and context is not None:
        return _get_token_async(
            agent_id=agent_id,
            tenant_id=tenant_id,
            auth=auth,
            auth_handler_name=auth_handler_name,
            context=context,
            scopes=normalized_scopes,
        )

    return _get_cached_token(agent_id, tenant_id, normalized_scopes) or ""

Return whether a non-expiring cached token exists for the agent and scopes.

In [5]:
def token_is_valid(
    *,
    agent_id: str | None,
    tenant_id: str | None = None,
    scopes: Sequence[str] | str | None = None,
) -> bool:
    normalized_scopes = _normalize_scopes(scopes)
    return _get_cached_token(agent_id, tenant_id, normalized_scopes) is not None

Exchange and cache an agentic token when no valid cached token exists.

In [6]:
async def _get_token_async(
    *,
    agent_id: str | None,
    tenant_id: str | None,
    auth: Any,
    auth_handler_name: str | None,
    context: Any,
    scopes: tuple[str, ...],
) -> str:
    cached_token = _get_cached_token(agent_id, tenant_id, scopes)
    if cached_token:
        return cached_token

    if not agent_id:
        logger.warning("Cannot exchange token without agent_id")
        return ""

    if not auth_handler_name:
        logger.debug("Skipping token exchange because no auth handler is configured")
        return ""

    cache_key = _get_cache_key(agent_id, tenant_id, scopes)
    lock = _agent_locks.setdefault(cache_key, asyncio.Lock())
    async with lock:
        cached_token = _get_cached_token(agent_id, tenant_id, scopes)
        if cached_token:
            return cached_token

        logger.info(
            "Exchanging agentic token (tenant_id=%s, agent_id=%s)",
            tenant_id or "(unknown)",
            agent_id,
        )
        exchanged_token = await auth.exchange_token(
            context,
            scopes=list(scopes),
            auth_handler_id=auth_handler_name,
        )
        token = _extract_token_value(exchanged_token)
        expires_at = _extract_expires_at(exchanged_token, token)
        _agentic_token_cache[cache_key] = CachedToken(
            token=token,
            expires_at=expires_at,
            scopes=scopes,
            tenant_id=tenant_id,
        )
        logger.info(
            "Cached agentic token (tenant_id=%s, agent_id=%s, expires_in=%ss)",
            tenant_id or "(unknown)",
            agent_id,
            max(0, int(expires_at - time.time())),
        )
        return token

Store an agentic token in the in-memory cache.

In [7]:
def cache_agentic_token(
    tenant_id: str | None,
    agent_id: str,
    token: str,
    scopes: Sequence[str] | str | None = None,
) -> None:
    normalized_scopes = _normalize_scopes(scopes)
    cache_key = _get_cache_key(agent_id, tenant_id, normalized_scopes)
    _agentic_token_cache[cache_key] = CachedToken(
        token=token,
        expires_at=_extract_expires_at(None, token),
        scopes=normalized_scopes,
        tenant_id=tenant_id,
    )
    logger.debug("Cached agentic token for agent_id=%s", agent_id)

Return the cached agentic token for a tenant and agent if available.

In [8]:
def get_cached_agentic_token(tenant_id: str | None, agent_id: str) -> str | None:
    token = get_token(agent_id=agent_id, tenant_id=tenant_id)
    return token or None

Return a valid cached token for an agent and scope set.

In [9]:
def _get_cached_token(agent_id: str | None, tenant_id: str | None, scopes: tuple[str, ...]) -> str | None:
    if not agent_id:
        logger.warning("Cannot resolve cached token without agent_id")
        return None

    cache_key = _get_cache_key(agent_id, tenant_id, scopes)
    cached_token = _agentic_token_cache.get(cache_key)
    if not cached_token:
        logger.debug("No cached token found for agent_id=%s", agent_id)
        return None

    if not cached_token.is_valid_for(scopes):
        logger.info("Cached token needs refresh for agent_id=%s", agent_id)
        return None

    logger.debug("Using cached token for agent_id=%s", agent_id)
    return cached_token.token

Build a cache key that isolates tenants and token audiences.

In [10]:
def _get_cache_key(agent_id: str, tenant_id: str | None, scopes: tuple[str, ...]) -> str:
    scope_key = " ".join(scopes)
    return f"{tenant_id or ''}:{agent_id}:{scope_key}"

Normalize scope input into a tuple of strings.

In [11]:
def _normalize_scopes(scopes: Sequence[str] | str | None) -> tuple[str, ...]:
    if scopes is None:
        return ()
    if isinstance(scopes, str):
        return (scopes,)
    return tuple(scopes)

Extract the raw token string from an exchange result.

In [12]:
def _extract_token_value(exchanged_token: Any) -> str:
    if isinstance(exchanged_token, str):
        return exchanged_token

    token = getattr(exchanged_token, "token", None)
    if isinstance(token, str) and token:
        return token

    raise ValueError("Token exchange did not return a token value")

Resolve the absolute expiration time for an exchanged token.

In [13]:
def _extract_expires_at(exchanged_token: Any, token: str) -> float:
    now = time.time()

    for attr in ("expires_on", "expires_at"):
        expires_at = getattr(exchanged_token, attr, None)
        if isinstance(expires_at, int | float):
            return float(expires_at)

    for attr in ("expires_in", "expires_in_seconds"):
        expires_in = getattr(exchanged_token, attr, None)
        if isinstance(expires_in, int | float):
            return now + float(expires_in)

    jwt_expires_at = _decode_jwt_exp(token)
    if jwt_expires_at is not None:
        return jwt_expires_at

    # Set expire to longer than buffer if token doesn't have expiry
    return now + TOKEN_REFRESH_BUFFER_SECONDS

Decode the expiration claim from a JWT token

In [14]:
def _decode_jwt_exp(token: str) -> float | None:
    parts = token.split(".")
    if len(parts) < 2:
        return None

    payload = parts[1]
    padded_payload = payload + "=" * (-len(payload) % 4)
    try:
        decoded = base64.urlsafe_b64decode(padded_payload.encode("ascii"))
        claims = json.loads(decoded)
    except (binascii.Error, UnicodeDecodeError, json.JSONDecodeError, ValueError):
        return None

    exp = claims.get("exp")
    if isinstance(exp, int | float):
        return float(exp)
    return None

## Agent
Implements a LangChain-backed assistant with MCP tool support.

In [15]:
import logging
from abc import ABC, abstractmethod
from os import environ
from typing import Optional

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, before_model
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, trim_messages
from langchain_core.tools import BaseTool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver
from azure.identity import ManagedIdentityCredential
from microsoft_agents.hosting.core import Authorization, TurnContext
from microsoft_agents_a365.notifications.agent_notification import NotificationTypes
from token_manager import get_token, token_is_valid

Declares the lifecycle and messaging contract for agent implementations.

In [16]:
class AgentInterface(ABC):

    @abstractmethod
    async def initialize(self) -> None:
        """Prepare the agent before it starts handling turns."""
        pass

    @abstractmethod
    async def process_user_message(
        self, message: str, auth: Authorization, auth_handler_name: Optional[str], context: TurnContext
    ) -> str:
        """Process a user message and return the response text."""
        pass

    @abstractmethod
    async def cleanup(self) -> None:
        """Release resources owned by the agent implementation."""
        pass


def check_agent_inheritance(agent_class) -> bool:
    """Return whether the supplied class implements AgentInterface."""
    if not issubclass(agent_class, AgentInterface):
        print(f"❌ Agent {agent_class.__name__} does not inherit from AgentInterface")
        return False
    return True

Keep persisted conversation history within the model context window.

In [17]:
@before_model
def trim_conversation_history(state: AgentState, runtime) -> dict:
    return {
        "messages": trim_messages(
            state["messages"],
            strategy="last",
            token_counter="approximate",
            max_tokens=12_000,
            start_on="human",
            include_system=True,
        )
    }

Runs user messages and notifications through a LangChain agent.

In [18]:
class LangChainAgent(AgentInterface):

    AGENT_PROMPT = environ.get("AGENT_PROMPT", "You are a helpful assistant.")

    def __init__(self):
        """Create the chat model, agent, and supporting services."""
        self.logger = logging.getLogger(self.__class__.__name__)
        self.model = self._create_model()
        self.tools: list[BaseTool] = []
        self.checkpointer = InMemorySaver()
        self.agent = self._create_agent(self.tools)

    def _create_model(self):
        """Create the LangChain chat model instance."""
        return init_chat_model(
            f"azure_ai:{environ['FOUNDRY_MODEL']}",
            project_endpoint=environ['FOUNDRY_PROJECT_ENDPOINT'],
            credential=ManagedIdentityCredential(client_id=environ['UAMI_CLIENT_ID']),
        )

    def _create_agent(self, tools: list[BaseTool]):
        """Create the LangChain agent instance."""
        agent = create_agent(
            model=self.model,
            tools=tools,
            system_prompt=self.AGENT_PROMPT,
            middleware=[trim_conversation_history],
            checkpointer=self.checkpointer,
        )
        logger.info("✅ LangChain agent created with %d tools", len(tools))
        return agent

    async def _setup_mcp_servers(self, auth: Authorization, auth_handler_name: Optional[str], context: TurnContext):
        """Attach configured MCP tool servers to the agent once per process."""
        tenant_id = getattr(context.activity.recipient, "tenant_id", None)
        agent_id = getattr(context.activity.recipient, "agentic_app_id", None)

        try:
            client = MultiServerMCPClient(
                {
                    "sentinel-mcp-data-exploration": {
                        "transport": "streamable_http",
                        "url": "https://sentinel.microsoft.com/mcp/data-exploration",
                        "headers": {
                            "Authorization": f"Bearer {await get_token(
                                    tenant_id=tenant_id,
                                    agent_id=agent_id,
                                    auth=auth,
                                    auth_handler_name=auth_handler_name,
                                    context=context,
                                    scopes="4500ebfb-89b6-4b14-a480-7f749797bfcd/.default",
                                )
                            }"
                        }
                    },
                    "sentinel-mcp-defender-triage": {
                        "transport": "streamable_http",
                        "url": "https://sentinel.microsoft.com/mcp/triage",
                        "headers": {
                            "Authorization": f"Bearer {await get_token(
                                    tenant_id=tenant_id,
                                    agent_id=agent_id,
                                    auth=auth,
                                    auth_handler_name=auth_handler_name,
                                    context=context,
                                    scopes="https://mcp.securitycenter.microsoft.com/MCP.Read.All",
                                )
                            }"
                        }
                    }
                }
            )
            self.tools = await client.get_tools()
            self.agent = self._create_agent(self.tools)

            if self.agent:
                logger.info("✅ MCP setup completed")
                self.mcp_servers_initialized = True
            else:
                logger.warning("⚠️ MCP setup failed")

        except Exception as e:
            logger.error(f"MCP setup error: {e}")

    async def initialize(self):
        """Log that the agent has completed startup initialization."""
        logger.info("Agent initialized")

    async def process_user_message(
        self, message: str, auth: Authorization, auth_handler_name: Optional[str], context: TurnContext
    ) -> str:
        """Run a user message through the agent and return its response."""
        from_prop = context.activity.from_property
        logger.info(
            "Turn received from user — DisplayName: '%s', UserId: '%s', AadObjectId: '%s'",
            getattr(from_prop, "name", None) or "(unknown)",
            getattr(from_prop, "id", None) or "(unknown)",
            getattr(from_prop, "aad_object_id", None) or "(none)",
        )

        try:
            await self._setup_mcp_servers(auth, auth_handler_name, context)
            result = await self._invoke(message, context)
            return self._extract_result(result) or "I couldn't process your request at this time."
        except Exception as e:
            logger.error(f"Error processing message: {e}")
            return f"Sorry, I encountered an error: {str(e)}"

    async def handle_agent_notification_activity(
        self, notification_activity, auth: Authorization, auth_handler_name: Optional[str], context: TurnContext
    ) -> str:
        """Process a Microsoft 365 notification activity through the agent."""
        try:
            notification_type = notification_activity.notification_type
            logger.info(f"📬 Processing notification: {notification_type}")

            await self._setup_mcp_servers(auth, auth_handler_name, context)

            if notification_type == NotificationTypes.EMAIL_NOTIFICATION:
                if not hasattr(notification_activity, "email") or not notification_activity.email:
                    return "I could not find the email notification details."

                email = notification_activity.email
                email_body = getattr(email, "html_body", "") or getattr(email, "body", "")

                result = await self._invoke(email_body, context)
                return self._extract_result(result) or "Email notification processed."

            elif notification_type == NotificationTypes.WPX_COMMENT:
                if not hasattr(notification_activity, "wpx_comment") or not notification_activity.wpx_comment:
                    return "I could not find the Word notification details."

                wpx = notification_activity.wpx_comment
                doc_id = getattr(wpx, "document_id", "")
                comment_id = getattr(wpx, "initiating_comment_id", "")
                drive_id = "default"

                doc_message = f"You have a new comment on the Word document with id '{doc_id}', comment id '{comment_id}', drive id '{drive_id}'. Please retrieve the Word document as well as the comments and return it in text format."
                doc_result = await self._invoke(doc_message, context)
                word_content = self._extract_result(doc_result)

                comment_text = notification_activity.text or ""
                response_message = f"You have received the following Word document content and comments. Please refer to these when responding to comment '{comment_text}'. {word_content}"
                result = await self._invoke(response_message, context)
                return self._extract_result(result) or "Word notification processed."

            else:
                notification_message = notification_activity.text or f"Notification received: {notification_type}"
                result = await self._invoke(notification_message, context)
                return self._extract_result(result) or "Notification processed successfully."

        except Exception as e:
            logger.error(f"Error processing notification: {e}")
            return f"Sorry, I encountered an error processing the notification: {str(e)}"

    async def _invoke(self, message: str, context: TurnContext):
        """Invoke the agent within the checkpointed conversation thread."""
        return await self.agent.ainvoke(
            {"messages": [HumanMessage(content=message)]},
            config={"configurable": {"thread_id": self._get_thread_id(context)}},
        )

    def _get_thread_id(self, context: TurnContext) -> str:
        """Return a stable checkpoint namespace for the current conversation."""
        activity = context.activity
        conversation = getattr(activity, "conversation", None)
        conversation_id = getattr(conversation, "id", None) or getattr(activity, "conversation_id", None)
        if conversation_id:
            return str(conversation_id)

        sender = getattr(activity, "from_property", None)
        sender_id = getattr(sender, "id", None)
        return str(sender_id or getattr(activity, "id", "default"))

    def _extract_result(self, result) -> str:
        """Convert an agent run result object into response text."""
        if not result:
            return ""
        if isinstance(result, dict):
            messages = result.get("messages")
            if messages:
                return self._extract_message_text(messages[-1])
            output = result.get("output")
            if output is not None:
                return str(output)
        if hasattr(result, "contents"):
            return str(result.contents)
        elif hasattr(result, "text"):
            return str(result.text)
        elif hasattr(result, "content"):
            return str(result.content)
        else:
            return str(result)

    def _extract_message_text(self, message: BaseMessage | object) -> str:
        """Extract text from a LangChain message."""
        content = getattr(message, "content", message)
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            text_parts = []
            for part in content:
                if isinstance(part, str):
                    text_parts.append(part)
                elif isinstance(part, dict) and isinstance(part.get("text"), str):
                    text_parts.append(part["text"])
            return "\n".join(text_parts)
        return str(content)

    async def cleanup(self) -> None:
        """
        Used if agent uses McpToolRegistrationService from mcp_tool_registration_service.
        So now it's just a placeholder to fulfill AgentInterface contract since tools are managed by the MCP client and don't require explicit cleanup in this agent.
        try:
            if hasattr(self, "tool_service") and self.tool_service:
                await self.tool_service.cleanup()
            logger.info("Agent cleanup completed")
        except Exception as e:
            logger.error(f"Cleanup error: {e}")
        """
        return

## Server
Hosts a LangChain agent behind the Microsoft Agents HTTP server.

In [19]:
import asyncio
import logging
import socket
import sys
from os import environ

from aiohttp.web import Application, Request, Response, json_response, run_app
from aiohttp.web_middlewares import middleware as web_middleware
from agent import AgentInterface, LangChainAgent, check_agent_inheritance
from microsoft_agents.activity import load_configuration_from_env
from microsoft_agents.authentication.msal import MsalConnectionManager
from microsoft_agents.hosting.aiohttp import (
    CloudAdapter,
    jwt_authorization_middleware,
    start_agent_process,
)
from microsoft_agents.hosting.core import (
    AgentApplication,
    AgentAuthConfiguration,
    AuthenticationConstants,
    Authorization,
    ClaimsIdentity,
    MemoryStorage,
    TurnContext,
    TurnState,
)
from microsoft_agents_a365.notifications import EmailResponse
from microsoft_agents_a365.notifications.agent_notification import (
    AgentNotification,
    AgentNotificationActivity,
    ChannelId,
    NotificationTypes,
)
from microsoft.opentelemetry import use_microsoft_opentelemetry
from microsoft.opentelemetry.a365.core import BaggageBuilder
from microsoft.opentelemetry.a365.runtime import get_observability_authentication_scope
from token_manager import get_token

ms_agents_logger = logging.getLogger("microsoft_agents")
ms_agents_logger.addHandler(logging.StreamHandler())
ms_agents_logger.setLevel(logging.INFO)

observability_logger = logging.getLogger("microsoft_agents_a365.observability")
observability_logger.setLevel(logging.ERROR)

logger = logging.getLogger(__name__)

agents_sdk_config = load_configuration_from_env(environ)

Create the generic host and run it with the supplied agent class.

In [20]:
def create_and_run_host(agent_class: type[AgentInterface], *agent_args, **agent_kwargs):
    if not check_agent_inheritance(agent_class):
        raise TypeError(f"Agent class {agent_class.__name__} must inherit from AgentInterface")

    use_microsoft_opentelemetry(
        enable_a365=True,
        a365_token_resolver=lambda agent_id, tenant_id: get_token(
            agent_id=agent_id,
            tenant_id=tenant_id,
            scopes=get_observability_authentication_scope(),
        ),
        instrumentation_options={
            "openai": {"enabled": False},
            "openai_agents": {"enabled": False},
            "langchain": {"enabled": True},
            "semantic_kernel": {"enabled": False},
            "agent_framework": {"enabled": False},
        },
    )

    host = GenericAgentHost(agent_class, *agent_args, **agent_kwargs)
    auth_config = host.create_auth_configuration()
    host.start_server(auth_config)

Configures authentication, handlers, notifications, and HTTP hosting.

In [21]:
class GenericAgentHost:

    def __init__(self, agent_class: type[AgentInterface], *agent_args, **agent_kwargs):
        """Initialize host services and register activity handlers."""
        if not check_agent_inheritance(agent_class):
            raise TypeError(f"Agent class {agent_class.__name__} must inherit from AgentInterface")

        self.auth_handler_name = environ.get("AUTH_HANDLER_NAME", "") or None
        if self.auth_handler_name:
            logger.info(f"🔐 Using auth handler: {self.auth_handler_name}")
        else:
            logger.info("🔓 No auth handler configured (AUTH_HANDLER_NAME not set)")

        self.agent_class = agent_class
        self.agent_args = agent_args
        self.agent_kwargs = agent_kwargs
        self.agent_instance = None

        self.storage = MemoryStorage()
        self.connection_manager = MsalConnectionManager(**agents_sdk_config)
        self.adapter = CloudAdapter(connection_manager=self.connection_manager)
        self.authorization = Authorization(
            self.storage, self.connection_manager, **agents_sdk_config
        )
        self.agent_app = AgentApplication[TurnState](
            storage=self.storage,
            adapter=self.adapter,
            authorization=self.authorization,
            **agents_sdk_config,
        )
        self.agent_notification = AgentNotification(self.agent_app)
        self._setup_handlers()
        logger.info("✅ Notification handlers registered successfully")

    async def _setup_observability_token(
        self, context: TurnContext, tenant_id: str, agent_id: str
    ):
        """Prepare the observability token for the current turn."""
        try:
            token = await get_token(
                agent_id=agent_id,
                tenant_id=tenant_id,
                auth=self.agent_app.auth,
                auth_handler_name=self.auth_handler_name,
                context=context,
                scopes=get_observability_authentication_scope(),
            )
            if token:
                logger.info(
                    f"✅ Observability token ready "
                    f"(tenant_id={tenant_id}, agent_id={agent_id})"
                )
        except Exception as e:
            logger.warning(f"⚠️ Failed to prepare observability token: {e}")

    async def _validate_agent_and_setup_context(self, context: TurnContext):
        """Validate that an agent is ready and return tenant and agent identifiers."""
        logger.info("🔍 Validating agent and setting up context...")
        tenant_id = context.activity.recipient.tenant_id
        agent_id = context.activity.recipient.agentic_app_id
        logger.info(f"🔍 tenant_id={tenant_id}, agent_id={agent_id}")

        if not self.agent_instance:
            logger.error("Agent not available")
            await context.send_activity("❌ Sorry, the agent is not available.")
            return None

        await self._setup_observability_token(context, tenant_id, agent_id)
        return tenant_id, agent_id

    def _setup_handlers(self):
        """Register conversation, message, installation, and notification handlers."""
        handler_config = {"auth_handlers": [self.auth_handler_name]} if self.auth_handler_name else {}

        async def help_handler(context: TurnContext, _: TurnState):
            """Send a short help message for welcome and help activities."""
            await context.send_activity(
                f"👋 **Hi there!** I'm **{self.agent_class.__name__}**, your AI assistant.\n\n"
                "How can I help you today?"
            )

        self.agent_app.conversation_update("membersAdded", **handler_config)(help_handler)
        self.agent_app.message("/help", **handler_config)(help_handler)

        @self.agent_app.activity("installationUpdate")
        async def on_installation_update(context: TurnContext, _: TurnState):
            """Handle agent installation and removal updates."""
            action = context.activity.action
            from_prop = context.activity.from_property
            logger.info(
                "InstallationUpdate received — Action: '%s', DisplayName: '%s', UserId: '%s'",
                action or "(none)",
                getattr(from_prop, "name", "(unknown)") if from_prop else "(unknown)",
                getattr(from_prop, "id", "(unknown)") if from_prop else "(unknown)",
            )
            if action == "add":
                await context.send_activity(
                    "Thank you for hiring me! Looking forward to assisting you in your professional journey!"
                )
            elif action == "remove":
                await context.send_activity(
                    "Thank you for your time, I enjoyed working with you."
                )

        @self.agent_app.activity("message", **handler_config)
        async def on_message(context: TurnContext, _: TurnState):
            """Handle incoming chat messages from users."""
            try:
                result = await self._validate_agent_and_setup_context(context)
                if result is None:
                    return
                tenant_id, agent_id = result

                with BaggageBuilder().tenant_id(tenant_id).agent_id(agent_id).build():
                    user_message = context.activity.text.strip() or ""
                    if not user_message or user_message == "/help":
                        return
                    logger.info(f"📨 {getattr(context.activity, 'channel_id')}: {user_message}")

                    response = await self.agent_instance.process_user_message(
                        user_message, self.agent_app.auth, self.auth_handler_name, context
                    )
                    await context.send_activity(response)

            except Exception as e:
                logger.error(f"❌ Error: {e}")
                await context.send_activity(f"Sorry, I encountered an error: {str(e)}")

        @self.agent_notification.on_agent_notification(
            channel_id=ChannelId(channel="agents", sub_channel="*"),
            **handler_config,
        )
        async def on_notification(
            context: TurnContext,
            state: TurnState,
            notification_activity: AgentNotificationActivity,
        ):
            """Handle incoming Microsoft 365 notification activities."""
            try:
                result = await self._validate_agent_and_setup_context(context)
                if result is None:
                    return
                tenant_id, agent_id = result

                with BaggageBuilder().tenant_id(tenant_id).agent_id(agent_id).build():
                    logger.info(f"📬 {notification_activity.notification_type}")

                    if not hasattr(self.agent_instance, "handle_agent_notification_activity"):
                        logger.warning("⚠️ Agent doesn't support notifications")
                        await context.send_activity(
                            "This agent doesn't support notification handling yet."
                        )
                        return

                    response = await self.agent_instance.handle_agent_notification_activity(
                        notification_activity, self.agent_app.auth, self.auth_handler_name, context
                    )

                    if notification_activity.notification_type == NotificationTypes.EMAIL_NOTIFICATION:
                        response_activity = EmailResponse.create_email_response_activity(response)
                        await context.send_activity(response_activity)
                        return

                    await context.send_activity(response)

            except Exception as e:
                logger.error(f"❌ Notification error: {e}")
                await context.send_activity(
                    f"Sorry, I encountered an error processing the notification: {str(e)}"
                )

    async def initialize_agent(self):
        """Create and initialize the agent instance on application startup."""
        if self.agent_instance is None:
            logger.info(f"🤖 Initializing {self.agent_class.__name__}...")
            self.agent_instance = self.agent_class(*self.agent_args, **self.agent_kwargs)
            await self.agent_instance.initialize()

    def create_auth_configuration(self) -> AgentAuthConfiguration | None:
        """Create the authentication configuration used by the host."""
        service_settings = agents_sdk_config["CONNECTIONS"]["SERVICE_CONNECTION"]["SETTINGS"]
        logger.info("🔒 AgentAuthConfiguration initialized with FIC")
        return AgentAuthConfiguration(
            **service_settings,
            scopes=["5a807f24-c9de-44ee-a3a7-329e88a00ffc/.default"],
        )

    def start_server(self, auth_configuration: AgentAuthConfiguration | None = None):
        """Build and run the aiohttp server for agent traffic."""

        async def entry_point(req: Request) -> Response:
            """Forward message requests into the Microsoft Agents adapter."""
            return await start_agent_process(req, req.app["agent_app"], req.app["adapter"])

        async def health(_req: Request) -> Response:
            """Return the health state of the hosted agent."""
            return json_response(
                {
                    "status": "ok",
                    "agent_type": self.agent_class.__name__,
                    "agent_initialized": self.agent_instance is not None,
                }
            )

        @web_middleware
        async def log_request(request, handler):
            """Log request headers and body before dispatching to a handler."""
            sensitive_headers = {"authorization", "cookie", "proxy-authorization"}
            headers = {
                name: "<redacted>" if name.lower() in sensitive_headers else value
                for name, value in request.headers.items()
            }
            body = (await request.read()).decode("utf-8", errors="replace")
            logger.info(
                "Incoming request: %s %s\nHeaders: %s\nBody: %s",
                request.method,
                request.path_qs,
                headers,
                body,
            )
            return await handler(request)

        middlewares = [log_request]
        if auth_configuration:

            @web_middleware
            async def jwt_with_health_bypass(request, handler):
                """Apply JWT authorization except for health checks."""
                if request.path == "/api/health":
                    return await handler(request)
                return await jwt_authorization_middleware(request, handler)

            middlewares.append(jwt_with_health_bypass)

        @web_middleware
        async def anonymous_claims(request, handler):
            """Inject anonymous claims when authentication is disabled."""
            if not auth_configuration:
                request["claims_identity"] = ClaimsIdentity(
                    {
                        AuthenticationConstants.AUDIENCE_CLAIM: "anonymous",
                        AuthenticationConstants.APP_ID_CLAIM: "anonymous-app",
                    },
                    False,
                    "Anonymous",
                )
            return await handler(request)

        middlewares.append(anonymous_claims)
        app = Application(middlewares=middlewares)

        app.router.add_post("/api/messages", entry_point)
        app.router.add_get("/api/messages", lambda _: Response(status=200))
        app.router.add_get("/api/health", health)

        app["agent_configuration"] = auth_configuration
        app["agent_app"] = self.agent_app
        app["adapter"] = self.agent_app.adapter

        app.on_startup.append(lambda app: self.initialize_agent())
        app.on_shutdown.append(lambda app: self.cleanup())

        host = environ.get("HOST", "localhost")
        port = int(environ.get("PORT", 3978))

        print("=" * 80)
        print(f"🏢 {self.agent_class.__name__}")
        print("=" * 80)
        print(f"🔒 Auth: {'Enabled' if auth_configuration else 'Anonymous'}")
        print(f"🚀 Server: {host}:{port}")
        print(f"📚 Endpoint: http://{host}:{port}/api/messages")
        print(f"❤️  Health: http://{host}:{port}/api/health\n")

        try:
            run_app(app, host=host, port=port, handle_signals=True)
        except KeyboardInterrupt:
            print("\n👋 Server stopped")

    async def cleanup(self):
        """Clean up the hosted agent instance during shutdown."""
        if self.agent_instance:
            try:
                await self.agent_instance.cleanup()
            except Exception as e:
                logger.error(f"Cleanup error: {e}")

Start the generic host and return the process exit code.

In [22]:
def main() -> int:
    try:
        print("Starting Generic Agent Host with LangChainAgent...")
        print()
        create_and_run_host(LangChainAgent)
    except Exception as e:
        print(f"❌ Failed to start server: {e}")
        import traceback

        traceback.print_exc()
        return 1

    return 0

In [23]:
if __name__ == "__main__":
    sys.exit(main())

INFO:__main__:🔐 Using auth handler: AGENTIC

Connections:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "CLIENTID": "dab...",
    "TENANTID": "323...",
    "AUTHORITY": "https://login.microsoftonline.com/...",
    "SCOPES": "... [36 scope(s)]",
    "FEDERATED_CLIENT_ID": "43e...",
    "ANONYMOUS_ALLOWED": "False"
  }
]

Connections Map:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "SERVICEURL": "*"
  }
]
INFO:microsoft_agents.hosting.core.authorization.connection_manager:
Connections:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "CLIENTID": "dab...",
    "TENANTID": "323...",
    "AUTHORITY": "https://login.microsoftonline.com/...",
    "SCOPES": "... [36 scope(s)]",
    "FEDERATED_CLIENT_ID": "43e...",
    "ANONYMOUS_ALLOWED": "False"
  }
]

Connections Map:
[
  {
    "CONNECTION": "SERVICE_CONNECTION",
    "SERVICEURL": "*"
  }
]
INFO:__main__:✅ Notification handlers registered successfully
INFO:__main__:🔒 AgentAuthConfiguration initialized with FIC


Starting Generic Agent Host with LangChainAgent...

🏢 LangChainAgent
🔒 Auth: Enabled
🚀 Server: 0.0.0.0:3978
📚 Endpoint: http://0.0.0.0:3978/api/messages
❤️  Health: http://0.0.0.0:3978/api/health

❌ Failed to start server: Cannot run the event loop while another loop is running


Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/site-packages/aiohttp/web.py", line 502, in run_app
    loop.run_until_complete(main_task)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/asyncio/base_events.py", line 701, in run_until_complete
    self._check_running()
    ~~~~~~~~~~~~~~~~~~~^^
  File "/opt/conda/lib/python3.13/asyncio/base_events.py", line 639, in _check_running
    raise RuntimeError(
        'Cannot run the event loop while another loop is running')
RuntimeError: Cannot run the event loop while another loop is running

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/site-packages/aiohttp/web.py", line 518, in run_app
    loop.run_until_complete(main_task)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/asyncio/base_events.py", line 701, in run_until_complete
    self._check_running()
    ~~~~~~~~~~~~~~~~~~~^

SystemExit: 1

/opt/conda/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
